# Chapter 68 — The Full Tiny GPT Architecture

The previous chapters developed embeddings, causal attention, feedforward networks, and complete transformer blocks.

This chapter assembles those pieces into a small causal language model that maps token IDs to next-token logits.


## Learning goals

By the end of this chapter, you will be able to:

- trace the full path from token IDs to vocabulary logits;
- implement a configurable `TinyGPT` class;
- explain every major model shape;
- explain why the model returns logits at every position;
- construct shifted next-token targets;
- compute cross-entropy over all batch positions;
- count parameters by architectural component; and
- distinguish the model architecture from training and generation loops.


## From token IDs to logits

A **logit** is a raw, unnormalized score for one possible next token.

A GPT-style model follows this path:

```text
token IDs                         [B, T]
  → token embeddings              [B, T, C]
  + position embeddings           [T, C]
  → combined embeddings           [B, T, C]
  → embedding dropout             [B, T, C]
  → transformer block stack       [B, T, C]
  → final layer normalization     [B, T, C]
  → vocabulary output layer       [B, T, V]
```

Here `B` is batch size, `T` is current context length, `C` is embedding dimension, and `V` is vocabulary size.

The model preserves batch and position dimensions while the final layer changes only `C` into `V`.


## Build a character vocabulary

A **vocabulary** is the set of token IDs the model can receive and predict.

We will use characters as tokens so every encoding step remains visible.


In [1]:
import torch

device = "cpu"
training_text = "the cat sat on the mat."
characters = sorted(set(training_text))
character_to_id = {
    character: character_id for character_id, character in enumerate(characters)
}
id_to_character = {
    character_id: character for character, character_id in character_to_id.items()
}

print("device:", device)
print("training text:", repr(training_text))
print("vocabulary size:", len(characters))
print("token mapping:", character_to_id)

device: cpu
training text: 'the cat sat on the mat.'
vocabulary size: 11
token mapping: {' ': 0, '.': 1, 'a': 2, 'c': 3, 'e': 4, 'h': 5, 'm': 6, 'n': 7, 'o': 8, 's': 9, 't': 10}


The vocabulary contains one integer ID for each distinct character in the teaching text.


## Encode text and create shifted targets

An input position is trained to predict the token immediately after it.

The target sequence is therefore the input sequence shifted forward by one token.


In [2]:
def encode_text(text: str, token_to_id: dict[str, int]) -> list[int]:
    return [token_to_id[character] for character in text]


def decode_token_ids(token_ids: list[int], id_to_token: dict[int, str]) -> str:
    return "".join(id_to_token[token_id] for token_id in token_ids)


all_token_ids = torch.tensor(
    encode_text(training_text, character_to_id),
    dtype=torch.long,
    device=device,
)
batch_size = 2
context_length = 8
start_positions = [0, 4]
input_token_ids = torch.stack(
    [all_token_ids[start : start + context_length] for start in start_positions]
)
target_token_ids = torch.stack(
    [all_token_ids[start + 1 : start + context_length + 1] for start in start_positions]
)

print("batch | input text | target text")
print("-" * 48)
for batch_index in range(batch_size):
    input_text = decode_token_ids(
        input_token_ids[batch_index].tolist(), id_to_character
    )
    target_text = decode_token_ids(
        target_token_ids[batch_index].tolist(), id_to_character
    )
    print(f"{batch_index:>5} | {input_text!r:>12} | {target_text!r:>12}")

print("input shape:", input_token_ids.shape)
print("target shape:", target_token_ids.shape)

batch | input text | target text
------------------------------------------------
    0 |   'the cat ' |   'he cat s'
    1 |   'cat sat ' |   'at sat o'
input shape: torch.Size([2, 8])
target shape: torch.Size([2, 8])


Each of the `B × T = 16` input positions now has one next-character target.


## Choose the model configuration

The model uses clear names for every architectural choice.

The dropout rate is zero for deterministic instructional outputs, but the modules will support nonzero dropout during training.


In [3]:
vocabulary_size = len(characters)
embedding_dimension = 16
number_of_attention_heads = 4
number_of_transformer_blocks = 2
dropout_rate = 0.0
head_size = embedding_dimension // number_of_attention_heads

print("vocabulary_size:", vocabulary_size)
print("context_length:", context_length)
print("embedding_dimension:", embedding_dimension)
print("number_of_attention_heads:", number_of_attention_heads)
print("number_of_transformer_blocks:", number_of_transformer_blocks)
print("dropout_rate:", dropout_rate)
print("head_size:", head_size)

vocabulary_size: 11
context_length: 8
embedding_dimension: 16
number_of_attention_heads: 4
number_of_transformer_blocks: 2
dropout_rate: 0.0
head_size: 4


The equal-head design requires `embedding_dimension` to divide evenly by `number_of_attention_heads`.

Here each of the four heads produces four features.


## Implement causal attention

These attention classes extend the Chapter 67 teaching implementation with dropout on attention weights and on the projected multi-head output.

The explicit per-head `ModuleList` keeps the architecture visible, although production implementations usually combine these operations for efficiency.


In [4]:
import math


class SingleHeadCausalSelfAttention(torch.nn.Module):
    """Compute one head of scaled causal self-attention."""

    embedding_dimension: int
    head_size: int
    context_length: int
    query_projection: torch.nn.Linear
    key_projection: torch.nn.Linear
    value_projection: torch.nn.Linear
    attention_dropout: torch.nn.Dropout
    causal_mask: torch.Tensor

    def __init__(
        self,
        embedding_dimension: int,
        head_size: int,
        context_length: int,
        dropout_rate: float,
    ) -> None:
        super().__init__()
        self.embedding_dimension = embedding_dimension
        self.head_size = head_size
        self.context_length = context_length
        self.query_projection = torch.nn.Linear(
            embedding_dimension, head_size, bias=False
        )
        self.key_projection = torch.nn.Linear(
            embedding_dimension, head_size, bias=False
        )
        self.value_projection = torch.nn.Linear(
            embedding_dimension, head_size, bias=False
        )
        self.attention_dropout = torch.nn.Dropout(dropout_rate)
        self.register_buffer(
            "causal_mask",
            torch.tril(torch.ones(1, context_length, context_length, dtype=torch.bool)),
        )

    def forward(self, input_values: torch.Tensor) -> torch.Tensor:
        current_context_length = input_values.shape[-2]
        queries = self.query_projection(input_values)
        keys = self.key_projection(input_values)
        values = self.value_projection(input_values)
        scores = queries @ keys.transpose(-2, -1)
        scaled_scores = scores / math.sqrt(self.head_size)
        allowed = self.causal_mask[:, :current_context_length, :current_context_length]
        weights = torch.softmax(
            scaled_scores.masked_fill(~allowed, float("-inf")),
            dim=-1,
        )
        return self.attention_dropout(weights) @ values


class MultiHeadCausalSelfAttention(torch.nn.Module):
    """Run equal-sized causal heads and combine their outputs."""

    embedding_dimension: int
    number_of_attention_heads: int
    head_size: int
    context_length: int
    attention_heads: torch.nn.ModuleList
    output_projection: torch.nn.Linear
    output_dropout: torch.nn.Dropout

    def __init__(
        self,
        embedding_dimension: int,
        number_of_attention_heads: int,
        context_length: int,
        dropout_rate: float,
    ) -> None:
        super().__init__()

        if embedding_dimension % number_of_attention_heads != 0:
            raise ValueError(
                "embedding dimension must divide evenly by the head count."
            )

        self.embedding_dimension = embedding_dimension
        self.number_of_attention_heads = number_of_attention_heads
        self.head_size = embedding_dimension // number_of_attention_heads
        self.context_length = context_length
        self.attention_heads = torch.nn.ModuleList(
            [
                SingleHeadCausalSelfAttention(
                    embedding_dimension=embedding_dimension,
                    head_size=self.head_size,
                    context_length=context_length,
                    dropout_rate=dropout_rate,
                )
                for _ in range(number_of_attention_heads)
            ]
        )
        self.output_projection = torch.nn.Linear(
            number_of_attention_heads * self.head_size,
            embedding_dimension,
        )
        self.output_dropout = torch.nn.Dropout(dropout_rate)

    def forward(self, input_values: torch.Tensor) -> torch.Tensor:
        head_outputs = [
            attention_head(input_values) for attention_head in self.attention_heads
        ]
        concatenated = torch.cat(head_outputs, dim=-1)
        projected = self.output_projection(concatenated)
        return self.output_dropout(projected)

The causal mask prevents every position from reading later positions.

Dropout changes values only in training mode and never changes tensor shape.


## Implement the transformer block

The block retains the pre-normalized residual order from Chapter 67.

Its feedforward network uses ReLU for continuity with Chapter 66, while many larger GPT-family models use GELU or gated activations.


In [5]:
class PositionWiseFeedForward(torch.nn.Module):
    """Apply a shared C-to-4C-to-C network at every position."""

    network: torch.nn.Sequential

    def __init__(self, embedding_dimension: int, dropout_rate: float) -> None:
        super().__init__()
        hidden_dimension = 4 * embedding_dimension
        self.network = torch.nn.Sequential(
            torch.nn.Linear(embedding_dimension, hidden_dimension),
            torch.nn.ReLU(),
            torch.nn.Linear(hidden_dimension, embedding_dimension),
            torch.nn.Dropout(dropout_rate),
        )

    def forward(self, input_values: torch.Tensor) -> torch.Tensor:
        return self.network(input_values)


class TransformerBlock(torch.nn.Module):
    """Apply one pre-normalized causal transformer block."""

    first_layer_norm: torch.nn.LayerNorm
    attention: MultiHeadCausalSelfAttention
    second_layer_norm: torch.nn.LayerNorm
    feedforward: PositionWiseFeedForward

    def __init__(
        self,
        embedding_dimension: int,
        number_of_attention_heads: int,
        context_length: int,
        dropout_rate: float,
    ) -> None:
        super().__init__()
        self.first_layer_norm = torch.nn.LayerNorm(embedding_dimension)
        self.attention = MultiHeadCausalSelfAttention(
            embedding_dimension=embedding_dimension,
            number_of_attention_heads=number_of_attention_heads,
            context_length=context_length,
            dropout_rate=dropout_rate,
        )
        self.second_layer_norm = torch.nn.LayerNorm(embedding_dimension)
        self.feedforward = PositionWiseFeedForward(
            embedding_dimension=embedding_dimension,
            dropout_rate=dropout_rate,
        )

    def forward(self, input_values: torch.Tensor) -> torch.Tensor:
        attention_change = self.attention(self.first_layer_norm(input_values))
        values_after_attention = input_values + attention_change
        feedforward_change = self.feedforward(
            self.second_layer_norm(values_after_attention)
        )
        return values_after_attention + feedforward_change

Every block receives and returns `[B, T, C]`, which lets the model stack independent blocks.


## Assemble `TinyGPT`

The complete model adds input embeddings before the block stack and a vocabulary projection after it.

The final layer normalization belongs after the entire stack rather than inside every block.

The model returns raw logits and optionally computes an all-position language-modeling loss.


In [6]:
class TinyGPT(torch.nn.Module):
    """Map token ID sequences to next-token logits with a tiny causal GPT."""

    vocabulary_size: int
    context_length: int
    embedding_dimension: int
    number_of_attention_heads: int
    number_of_transformer_blocks: int
    dropout_rate: float
    token_embedding_table: torch.nn.Embedding
    position_embedding_table: torch.nn.Embedding
    embedding_dropout: torch.nn.Dropout
    transformer_blocks: torch.nn.ModuleList
    final_layer_norm: torch.nn.LayerNorm
    output_layer: torch.nn.Linear

    def __init__(
        self,
        vocabulary_size: int,
        context_length: int,
        embedding_dimension: int,
        number_of_attention_heads: int,
        number_of_transformer_blocks: int,
        dropout_rate: float,
    ) -> None:
        super().__init__()
        dimensions = (
            vocabulary_size,
            context_length,
            embedding_dimension,
            number_of_attention_heads,
            number_of_transformer_blocks,
        )
        if min(dimensions) <= 0:
            raise ValueError("all dimensions and counts must be positive.")
        if embedding_dimension % number_of_attention_heads != 0:
            raise ValueError(
                "embedding dimension must divide evenly by the head count."
            )
        if not 0.0 <= dropout_rate < 1.0:
            raise ValueError("dropout_rate must be in the interval [0, 1).")

        self.vocabulary_size = vocabulary_size
        self.context_length = context_length
        self.embedding_dimension = embedding_dimension
        self.number_of_attention_heads = number_of_attention_heads
        self.number_of_transformer_blocks = number_of_transformer_blocks
        self.dropout_rate = dropout_rate
        self.token_embedding_table = torch.nn.Embedding(
            vocabulary_size, embedding_dimension
        )
        self.position_embedding_table = torch.nn.Embedding(
            context_length, embedding_dimension
        )
        self.embedding_dropout = torch.nn.Dropout(dropout_rate)
        self.transformer_blocks = torch.nn.ModuleList(
            [
                TransformerBlock(
                    embedding_dimension=embedding_dimension,
                    number_of_attention_heads=number_of_attention_heads,
                    context_length=context_length,
                    dropout_rate=dropout_rate,
                )
                for _ in range(number_of_transformer_blocks)
            ]
        )
        self.final_layer_norm = torch.nn.LayerNorm(embedding_dimension)
        self.output_layer = torch.nn.Linear(embedding_dimension, vocabulary_size)

    def _validate_input(self, input_token_ids: torch.Tensor) -> None:
        if input_token_ids.ndim != 2:
            raise ValueError("input token IDs must have shape [batch, context].")
        if input_token_ids.shape[1] == 0:
            raise ValueError("input context must contain at least one token.")
        if input_token_ids.shape[1] > self.context_length:
            raise ValueError("input exceeds the configured context length.")
        if input_token_ids.dtype not in (torch.int32, torch.int64):
            raise TypeError("input token IDs must use an integer dtype.")

    def forward_with_intermediates(
        self, input_token_ids: torch.Tensor
    ) -> dict[str, torch.Tensor]:
        self._validate_input(input_token_ids)
        current_context_length = input_token_ids.shape[1]
        token_embeddings = self.token_embedding_table(input_token_ids)
        position_ids = torch.arange(
            current_context_length,
            dtype=torch.long,
            device=input_token_ids.device,
        )
        position_embeddings = self.position_embedding_table(position_ids)
        combined_embeddings = token_embeddings + position_embeddings
        current_values = self.embedding_dropout(combined_embeddings)

        intermediates = {
            "input_token_ids": input_token_ids,
            "token_embeddings": token_embeddings,
            "position_embeddings": position_embeddings,
            "combined_embeddings": combined_embeddings,
            "embeddings_after_dropout": current_values,
        }
        for block_index, block in enumerate(self.transformer_blocks):
            current_values = block(current_values)
            intermediates[f"block_{block_index}_output"] = current_values

        normalized_output = self.final_layer_norm(current_values)
        vocabulary_logits = self.output_layer(normalized_output)
        intermediates["normalized_output"] = normalized_output
        intermediates["vocabulary_logits"] = vocabulary_logits
        return intermediates

    def forward(
        self,
        input_token_ids: torch.Tensor,
        target_token_ids: torch.Tensor | None = None,
    ) -> tuple[torch.Tensor, torch.Tensor | None]:
        intermediates = self.forward_with_intermediates(input_token_ids)
        vocabulary_logits = intermediates["vocabulary_logits"]
        loss = None

        if target_token_ids is not None:
            if target_token_ids.shape != input_token_ids.shape:
                raise ValueError("targets must have the same shape as inputs.")
            if target_token_ids.dtype not in (torch.int32, torch.int64):
                raise TypeError("target token IDs must use an integer dtype.")

            flattened_logits = vocabulary_logits.reshape(-1, self.vocabulary_size)
            flattened_targets = target_token_ids.reshape(-1)
            loss = torch.nn.functional.cross_entropy(
                flattened_logits, flattened_targets
            )

        return vocabulary_logits, loss

`forward_with_intermediates` exposes each model stage without creating a second implementation of the architecture.

The ordinary `forward` method uses those same logits and adds loss computation only when targets are provided.


## Create the model

A fixed seed makes parameter initialization and all stored outputs reproducible.


In [7]:
torch.manual_seed(68)
tiny_gpt = TinyGPT(
    vocabulary_size=vocabulary_size,
    context_length=context_length,
    embedding_dimension=embedding_dimension,
    number_of_attention_heads=number_of_attention_heads,
    number_of_transformer_blocks=number_of_transformer_blocks,
    dropout_rate=dropout_rate,
).to(device)

print(tiny_gpt)

TinyGPT(
  (token_embedding_table): Embedding(11, 16)
  (position_embedding_table): Embedding(8, 16)
  (embedding_dropout): Dropout(p=0.0, inplace=False)
  (transformer_blocks): ModuleList(
    (0-1): 2 x TransformerBlock(
      (first_layer_norm): LayerNorm((16,), eps=1e-05, elementwise_affine=True)
      (attention): MultiHeadCausalSelfAttention(
        (attention_heads): ModuleList(
          (0-3): 4 x SingleHeadCausalSelfAttention(
            (query_projection): Linear(in_features=16, out_features=4, bias=False)
            (key_projection): Linear(in_features=16, out_features=4, bias=False)
            (value_projection): Linear(in_features=16, out_features=4, bias=False)
            (attention_dropout): Dropout(p=0.0, inplace=False)
          )
        )
        (output_projection): Linear(in_features=16, out_features=16, bias=True)
        (output_dropout): Dropout(p=0.0, inplace=False)
      )
      (second_layer_norm): LayerNorm((16,), eps=1e-05, elementwise_affine=True)
  

The model contains two independent transformer blocks followed by one final layer normalization and one vocabulary output layer.


## Inspect the full forward path

The next cell prints the shape after every major stage, including each transformer block.


In [8]:
inspection = tiny_gpt.forward_with_intermediates(input_token_ids)
vocabulary_logits, no_target_loss = tiny_gpt(input_token_ids)

print("stage".ljust(32), "shape")
print("-" * 54)
for stage_name, stage_values in inspection.items():
    print(stage_name.ljust(32), tuple(stage_values.shape))

print()
print("forward logits match inspection:", end=" ")
print(torch.allclose(vocabulary_logits, inspection["vocabulary_logits"]))
print("loss without targets:", no_target_loss)

stage                            shape
------------------------------------------------------
input_token_ids                  (2, 8)
token_embeddings                 (2, 8, 16)
position_embeddings              (8, 16)
combined_embeddings              (2, 8, 16)
embeddings_after_dropout         (2, 8, 16)
block_0_output                   (2, 8, 16)
block_1_output                   (2, 8, 16)
normalized_output                (2, 8, 16)
vocabulary_logits                (2, 8, 11)

forward logits match inspection: True
loss without targets: None


The token IDs begin at `[2, 8]`, hidden values remain `[2, 8, 16]`, and the output logits end at `[2, 8, 11]`.

The position embeddings have shape `[8, 16]` and broadcast across the two batch items during addition.


## One vocabulary score vector per position

The output is `[B, T, V]`, not `[B, V]`.

Applying the output layer to one final hidden vector should reproduce the corresponding slice of the full logits tensor.


In [9]:
batch_index = 0
position = 3
one_hidden_vector = inspection["normalized_output"][batch_index, position]
one_position_logits = tiny_gpt.output_layer(one_hidden_vector)
full_model_position_logits = vocabulary_logits[batch_index, position]
one_position_probabilities = torch.softmax(one_position_logits, dim=-1)

print("hidden vector shape:", one_hidden_vector.shape)
print("one-position logits shape:", one_position_logits.shape)
print("matches full model slice:", end=" ")
print(torch.allclose(one_position_logits, full_model_position_logits))
print("probability sum:", one_position_probabilities.sum().item())

hidden vector shape: torch.Size([16])
one-position logits shape: torch.Size([11])
matches full model slice: True
probability sum: 1.0


The output layer applies the same `C → V` transformation independently at every position.

The model returns logits because cross-entropy accepts raw scores directly.

Softmax is needed only when probabilities are required for interpretation or token selection.


## Compute loss from every position

Cross-entropy treats every batch-position pair as one classification example over the vocabulary.

Flattening `[B, T, V]` into `[B × T, V]` preserves all 16 prediction problems.

The following check also computes unreduced loss at each position and verifies that its mean equals the model loss.


In [10]:
vocabulary_logits, language_model_loss = tiny_gpt(input_token_ids, target_token_ids)
flattened_logits = vocabulary_logits.reshape(-1, vocabulary_size)
flattened_targets = target_token_ids.reshape(-1)
per_position_loss = torch.nn.functional.cross_entropy(
    vocabulary_logits.transpose(1, 2),
    target_token_ids,
    reduction="none",
)

print("logits shape:", vocabulary_logits.shape)
print("flattened logits shape:", flattened_logits.shape)
print("flattened targets shape:", flattened_targets.shape)
print("per-position loss shape:", per_position_loss.shape)
print("language-model loss:", language_model_loss.item())
print("mean per-position loss:", per_position_loss.mean().item())
print("all-position means match:", end=" ")
print(torch.allclose(language_model_loss, per_position_loss.mean()))

logits shape: torch.Size([2, 8, 11])
flattened logits shape: torch.Size([16, 11])
flattened targets shape: torch.Size([16])
per-position loss shape: torch.Size([2, 8])
language-model loss: 2.3537025451660156
mean per-position loss: 2.3537020683288574
all-position means match: True


The loss tensor before reduction has shape `[B, T]`.

Its mean matches the scalar loss, confirming that every position contributes equally in this example.


## Verify full-model causality

Changing the final input token must not alter logits at earlier positions.

This end-to-end check covers embeddings, both transformer blocks, final normalization, and the vocabulary projection.


In [11]:
changed_final_token_ids = input_token_ids.clone()
changed_final_token_ids[:, -1] = (changed_final_token_ids[:, -1] + 1) % vocabulary_size

original_logits, _ = tiny_gpt(input_token_ids)
changed_logits, _ = tiny_gpt(changed_final_token_ids)
logit_change_by_position = (changed_logits - original_logits).abs().sum(dim=-1)

print("absolute logit change by position:")
print(logit_change_by_position)
print("earlier logits unchanged:", end=" ")
print(torch.allclose(original_logits[:, :-1], changed_logits[:, :-1]))
print("final logits changed:", end=" ")
print(not torch.allclose(original_logits[:, -1], changed_logits[:, -1]))

absolute logit change by position:
tensor([[0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 5.7011],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 5.8572]],
       grad_fn=<SumBackward1>)
earlier logits unchanged: True
final logits changed: True


Only the final position changes because every attention head is causal and every feedforward network is position-wise.


## Count parameters by component

Grouping parameters shows where model capacity is stored more clearly than one long list of tensor names.


In [12]:
def count_parameters(module: torch.nn.Module) -> int:
    return sum(parameter.numel() for parameter in module.parameters())


component_parameter_counts = {
    "token embeddings": count_parameters(tiny_gpt.token_embedding_table),
    "position embeddings": count_parameters(tiny_gpt.position_embedding_table),
    "transformer blocks": count_parameters(tiny_gpt.transformer_blocks),
    "final layer norm": count_parameters(tiny_gpt.final_layer_norm),
    "vocabulary output": count_parameters(tiny_gpt.output_layer),
}

for component_name, parameter_count in component_parameter_counts.items():
    print(f"{component_name:22} {parameter_count:5}")

component_total = sum(component_parameter_counts.values())
model_total = count_parameters(tiny_gpt)
print("component total:", component_total)
print("model total:", model_total)
print("totals match:", component_total == model_total)

token embeddings         176
position embeddings      128
transformer blocks      6464
final layer norm          32
vocabulary output        187
component total: 6987
model total: 6987
totals match: True


Dropout modules and causal-mask buffers add no trainable parameters.

This model also leaves the token embedding and vocabulary output weights untied, although some GPT implementations share those weights.


## Training uses all positions, generation uses the last

Training normally uses the full `[B, T, V]` logits tensor.

Autoregressive generation needs the distribution for the token after the current context, so it selects only the final position.


In [13]:
final_position_logits = vocabulary_logits[:, -1, :]
final_position_probabilities = torch.softmax(final_position_logits, dim=-1)
most_likely_next_token_ids = final_position_probabilities.argmax(dim=-1)

print("all-position logits shape:", vocabulary_logits.shape)
print("final-position logits shape:", final_position_logits.shape)
print("most likely token IDs:", most_likely_next_token_ids.tolist())
print(
    "most likely characters:",
    [id_to_character[token_id] for token_id in most_likely_next_token_ids.tolist()],
)

all-position logits shape: torch.Size([2, 8, 11])
final-position logits shape: torch.Size([2, 11])
most likely token IDs: [7, 7]
most likely characters: ['n', 'n']


The untrained model's choices are not meaningful.

This cell demonstrates only which output slice a later generation loop will consume.


## Dropout and model mode

Dropout randomly removes selected activations only while a model is in training mode.

`model.train()` enables dropout, and `model.eval()` disables it.

The main model uses `dropout_rate = 0.0`, so its outputs remain deterministic in either mode.

A later training loop can choose a nonzero rate, while generation should call `eval()` to disable dropout.


## Scope of this tiny architecture

This is a complete teaching-scale causal transformer architecture, but it is not a reproduction of one specific production GPT.

Important simplifications include:

- explicit separate attention-head modules;
- ReLU instead of a GELU or gated feedforward activation;
- learned absolute position embeddings;
- no embedding-output weight tying; and
- no optimized attention kernels or key-value cache.

These choices make the architecture easier to inspect without changing its central causal language-modeling flow.


## Common mistakes

- Do not omit position embeddings.
- Do not pass a sequence longer than the learned position table.
- Do not remove causal masking from attention.
- Do not reduce `[B, T, V]` to one position during ordinary language-model training.
- Do not apply softmax before `cross_entropy`.
- Do not confuse attention softmax over positions with vocabulary softmax over tokens.
- Do not treat the untrained model's most likely token as a meaningful prediction.
- Do not forget to switch a model with nonzero dropout to evaluation mode for generation.


## Takeaways

`TinyGPT` maps `[B, T]` token IDs to `[B, T, V]` next-token logits.

Token and position embeddings create the initial hidden vectors.

A stack of causal transformer blocks updates those vectors while preserving `[B, T, C]`.

Final normalization and the output layer map each position from `C` features to `V` logits.

> The model outputs vocabulary logits for every position, not only the final position.

Training can therefore average cross-entropy across every batch-position pair in one forward pass.

Generation later consumes the final-position logits one step at a time.


## What comes next

The next chapter will train this tiny GPT end to end with sampled batches, backpropagation, an optimizer, and validation loss.

That training loop will turn the architecture's logits into learned next-character predictions.
